# DC-EGM for Education Level 1

## Dependencies, Packages & Roots

### Magics

In [ ]:
%reload_ext autoreload
%autoreload 2

### Project Paths

In [ ]:
from pathlib import Path
import sys
import importlib

# Change this depending on notebook location:
# 0 = notebook is 1 folder inside project root, e.g. dp_termpaper/edu1
# 1 = notebook is 2 folders inside project root, e.g. dp_termpaper/data/moments
# 2 = notebook is 3 folders inside project root
amount_of_levels = 0

DIR = Path.cwd().resolve().parents[amount_of_levels]

if str(DIR) not in sys.path:
    sys.path.insert(0, str(DIR))

import project_paths as pp
from project_imports import *

importlib.reload(pp)
# importlib.reload(pi)

print("Project root:", pp.DIR)

Project root: C:\Users\Nashw\Studiet\polit\2. Semester\DynamicProgramming\Thesis\dp_termpaper


## Load Data 

In [ ]:
FILE_UDD1 = pp.MOMENTS_DIR / "moments_udd1.txt"
FILE_UDD2 = pp.MOMENTS_DIR / "moments_udd2.txt"
FILE_UDD3 = pp.MOMENTS_DIR / "moments_udd3.txt"

MORTALITY = pp.DATA_DIR / "mortality.xlsx"

In [ ]:
PARAM_FILE = pp.STRUCTURAL_RESULTS_DIR / "optimized_params_udd1.txt"
params = load_params_txt(PARAM_FILE)
# params

In [ ]:
# Read CSV
df_edu = pd.read_csv(FILE_UDD1)
# read mortality and discard seoncd and third column
df_mort = pd.read_excel(MORTALITY, sheet_name="DOD", usecols=[0, 3])

# 2) standardize colomn name and remove _FREQ_ column
# Iterate over DataFrames (if there are multiple DataFrames to process)
for data_frame in [df_edu]:
    # Rename ALDER → age
    if "ALDER" in data_frame.columns:
        data_frame.rename(columns={"ALDER": "age"}, inplace=True)
    # Remove _FREQ_-column if it exists
    if "_FREQ_" in data_frame.columns:
        data_frame.drop(columns=["_FREQ_"], inplace=True)
# drop var_wage, skew_wage and pens
df_edu.drop(columns=["var_wage", "skew_wage", "pens"], inplace=True)
df_edu

,age,prob_work,hours_0,hours_1,hours_2,hours_3,hours_4,avg_wealth,work_work,nowork_nowork,avg_experience,avg_labor_income,avg_wage,avg_hours
0,30,0.740845,0.259155,0.118310,0.108099,0.108451,0.405986,0.505047,NaN,NaN,NaN,2.529835,0.001800,1380.869772
1,31,0.725525,0.274475,0.107122,0.098097,0.101236,0.419070,0.551522,0.672383,0.237816,NaN,2.676016,0.001856,1417.171444
2,32,0.728371,0.271629,0.103818,0.096127,0.099423,0.429003,0.566578,0.660332,0.238393,NaN,2.751088,0.001885,1431.146305
3,33,0.729473,0.270527,0.106531,0.085814,0.103696,0.433431,0.647082,0.669245,0.236301,NaN,2.815883,0.001922,1438.460389
4,34,0.722898,0.277102,0.098927,0.081216,0.099463,0.443292,0.674491,0.666112,0.239242,NaN,2.922359,0.001956,1462.447414
5,35,0.718883,0.281117,0.092892,0.079012,0.104561,0.442419,0.731305,0.669990,0.241433,9.311996,2.991115,0.001986,1473.127520
6,36,0.747882,0.252118,0.085640,0.076319,0.098294,0.487629,0.873596,0.678487,0.229618,11.223930,3.280031,0.002115,1514.846288
7,37,0.763328,0.236672,0.081438,0.069953,0.095184,0.516753,1.062263,0.710226,0.209453,12.188312,3.492342,0.002214,1543.759256
8,38,0.777520,0.222480,0.074984,0.068018,0.093487,0.541031,1.159197,0.729952,0.196546,12.992333,3.672453,0.002290,1568.129486
9,39,0.786896,0.213104,0.073461,0.062904,0.089849,0.560682,1.336997,0.745816,0.183934,13.763616,3.832563,0.002361,1585.524050


## Options for model - Choices and states

In [ ]:
n_periods = 55
choices = np.arange(5) # 5 choices

options = {
    "model_params": {
        #"quadrature_points_stochastic": 5, 
        "n_quad_points_stochastic": 5,
        "n_periods": n_periods,
        "choices": choices, # 4 choices
        "hours": jnp.array([0,250,750,1300,1900]), #list 
        "max_hours": 1500,
        "start_age": 30,
        "tax_threshold1": 0.480,
        "tax_threshold2": 5.698,
        "tax_base_rate": 0.38,
        "tax_top_rate": 0.5,
        "retirement_age": 67,
        "oap_base_amount":0.80328,
        "oap_max_supplement": 0.92940,
        "supp_threshold": 0.79300,
        "oap_threshold": 3.3592,
        "supp_reduction_rate": 0.309,
        "oap_reduction_rate": 0.3,
        "alpha1": 0.000417, # independently estimated parameter for survival probability
        "alpha2": 0.1,  # independently estimated parameter for survival probability
        "max_init_experience": 5,
        "max_ret_period": 45, # Age 75
        "min_ret_period": 30, # Age 60
    },
    "state_space": {
        "n_periods": n_periods,
        "choices": choices, # 4 choices
        "continuous_states": {
            "wealth": np.linspace(0, 50, 20),
            "experience": jnp.linspace(0, 1, 5).astype(float) # 1 experince grid point, if experience can only go up by a year - more points if it is a fraction based on hours worked
        },
        "exogenous_processes": {
            "survival": {
                "transition": prob_survival,
                "states": [0, 1],
            },
        },
    },
}

# ====================================================================================================================

# Structural Estimation

## Initiatal parameters - some are estimated above, the rest to be structuraly estimated

In [ ]:
PARAM_FILE = "optimized_params_udd1_test.txt"
params = load_params_txt(PARAM_FILE)
params

In [ ]:
model_initial = setup_model(
    options=options,
    state_space_functions=create_state_space_function_dict(),
    utility_functions=utility_functions,
    utility_functions_final_period=final_period_utility,
    budget_constraint=budget_dcegm_initial,
)

model_counter = setup_model(
    options=options,
    state_space_functions=create_state_space_function_dict(),
    utility_functions=utility_functions,
    utility_functions_final_period=final_period_utility,
    budget_constraint=budget_dcegm_counter_oap,
)

In [ ]:
# # validate model
# validate_exogenous_processes(model, params)

# ====================================================================================================================

## Initial values for simulating. 

In [ ]:
# Select number of individuals for simulation
n_individuals = 10000

seed = 132
key = jax.random.PRNGKey(0)
n = n_individuals                        



# distribution of initial choices
labels = jnp.array([0, 1, 2, 3, 4], dtype=jnp.int32)
probs  = jnp.array([0.259155, 0.118310, 0.108099, 0.108451, 0.405986])  # sums to 1.0

lagged_choice = jax.random.choice(
    key,
    a       = labels,
    shape   = (n,),
    p       = probs,
    replace = True
)
# set initial states for each individual
states_initial = {
    "period": jnp.zeros(n_individuals),       # Every individual starts at period 0 (age 30)
    "lagged_choice": lagged_choice,  # Every individual starts with choice 3 (work fulltime)
    "experience": jnp.full(n_individuals, 0.8).astype(float),  # Every individual starts with 5 years of experience
    "survival": jnp.ones(n_individuals), # Every individual starts with 1 (alive)
}

# Set wealth at beginning of period, which is the starting wealth for every individual. 
wealth_initial = jnp.full(n_individuals, 0.505047)   # Every individual starts with 59k wealth - to be adjusted based on the actual moments

## Simulate model

In [ ]:
sim_func_aux_init = get_sol_and_sim_func_for_model(
    model=model_initial,
    states_initial=states_initial,
    wealth_initial=wealth_initial,
    n_periods=options["state_space"]["n_periods"],
    seed=seed,
)


output_dict_aux_init = sim_func_aux_init(params)

df_sim_init = create_simulation_df(output_dict_aux_init["sim_dict"])

# hours_map = options["model_params"]["hours"]
hours_map = {0: 0, 1: 250, 2: 750, 3: 1300, 4: 1900}
# start_age = options["model_params"]["start_age"]
start_age = 30  # Or start_age = options["model_params"]["start_age"]


moments_sim_init = compute_simulation_moments(df_sim_init, start_age, hours_map)


In [ ]:
sim_func_aux_count = get_sol_and_sim_func_for_model(
    model=model_counter,
    states_initial=states_initial,
    wealth_initial=wealth_initial,
    n_periods=options["state_space"]["n_periods"],
    seed=seed,
)


output_dict_aux_count = sim_func_aux_count(params)

df_sim_count = create_simulation_df(output_dict_aux_count["sim_dict"])

# hours_map = options["model_params"]["hours"]
hours_map = {0: 0, 1: 250, 2: 750, 3: 1300, 4: 1900}
# start_age = options["model_params"]["start_age"]
start_age = 30  # Or start_age = options["model_params"]["start_age"]


moments_sim_count = compute_simulation_moments(df_sim_count, start_age, hours_map)

## Convert simulation results to DataFrame

## Plots

In [ ]:
edu = df_edu.copy().loc[df_edu.age <= 75]       # empirical
moments_sim_count_ci = compute_simulation_moments_with_ci(df_sim_count, start_age, hours_map)  # Call the function
moments_sim_count_ci = moments_sim_count_ci.loc[moments_sim_count_ci.age <= 75]  # Filter simulated + CIs

moments_sim_init_ci = compute_simulation_moments_with_ci(df_sim_init, start_age, hours_map)  # Call the function
moments_sim_init_ci = moments_sim_init_ci.loc[moments_sim_init_ci.age <= 75]  # Filter simulated + CIs

# print plots
plot_empirical_vs_simulated_with_ci(moments_sim_init_ci, moments_sim_count_ci, pp.SIM_PLOTS_DIR, out_subfolder="edu1")
plot_empirical_vs_simulated_with_ci(edu, moments_sim_count_ci, pp.SIM_PLOTS_DIR, out_subfolder="edu1")

moments_sim_init_ci.to_pickle (os.path.join(pp.SIM_RESULTS_DIR, "moments_init_edu1.pkl"))
moments_sim_count_ci.to_pickle(os.path.join(pp.SIM_RESULTS_DIR, "moments_cf_edu1.pkl"))


In [ ]:
metrics = ['avg_hours', 'avg_wealth', 'avg_consumption', 'prob_work']
df_diff = compute_counterfactual_diff(
    df_base = moments_sim_init_ci,
    df_cf   = moments_sim_count_ci,
    metrics = metrics
)

df_diff 

plot_cf_diff_separate(df_diff, metrics)